In [1]:
#!pip install -r requirements.txt

In [2]:
import torch
import pandas as pd
import feature_engineering
from dataset.dataset_construction import TimeSeriesDataset
from model import *

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
file_path = './dataset/dataset.csv'
raw_dataset_df = pd.read_csv(file_path)

df_hybrid = feature_engineering.featureEng(raw_dataset_df)

X_train, y_train, X_test, y_test = feature_engineering.createSplit(df_hybrid)

In [4]:
preprocessing = feature_engineering.createPreprocessingPipeline(X_train.columns)
X_train_processed = preprocessing.fit_transform(X_train)
X_test_processed  = preprocessing.transform(X_test)

# Drop the NaN rows created by lag shifts
X_train_processed = X_train_processed.dropna()
y_train = y_train.loc[X_train_processed.index]

X_test_processed = X_test_processed.dropna()
y_test = y_test.loc[X_test_processed.index]

In [5]:
train_df = pd.concat([X_train_processed, y_train], axis=1)
test_df  = pd.concat([X_test_processed,  y_test],  axis=1)

train_dataset = TimeSeriesDataset(train_df, input_width=7, label_width=1, shift=1, label_columns=['internacoes'])
test_dataset = TimeSeriesDataset(test_df, input_width=7, label_width=1, shift=1, label_columns=['internacoes'])

In [6]:
n_features = X_train_processed.shape[1]

hyperparameters = {
    'input_size': 30,
    'hidden_size': 64,
    'num_layers': 2,
    'output_size': 1,
    'label_width': 1,
    'dropout': 0.2,
    'batch_size': 32
}

model = LSTMModel(
    input_size=hyperparameters['input_size'],
    hidden_size=hyperparameters['hidden_size'],
    num_layers=hyperparameters['num_layers'],
    output_size=hyperparameters['output_size'],
    label_width=hyperparameters['label_width'],
    dropout=hyperparameters['dropout']
)

In [7]:
model, cv_history, final_history = train(
    model, train_dataset,
    hyperparameters=hyperparameters,
    n_splits=5,
    epochs=30,
    lr=1e-3,
    patience=5,
    device=device
)

Starting 5-fold time series cross-validation...

───────────────────────────────────────────────────────
Fold 1/5 | train: 203 samples | val: 200 samples
───────────────────────────────────────────────────────
  Epoch   1/30 | train_loss: 7.7773 | val_MSE: 21.0957 | val_MAE: 3.7529
  Epoch   2/30 | train_loss: 6.1307 | val_MSE: 16.6133 | val_MAE: 3.1721
  Epoch   3/30 | train_loss: 3.6358 | val_MSE: 10.2458 | val_MAE: 2.3309
  Epoch   4/30 | train_loss: 2.6059 | val_MSE: 8.3033 | val_MAE: 2.0948
  Epoch   5/30 | train_loss: 2.6214 | val_MSE: 9.1955 | val_MAE: 2.2181
  Epoch   6/30 | train_loss: 2.4488 | val_MSE: 10.1655 | val_MAE: 2.3233
  Epoch   7/30 | train_loss: 2.4711 | val_MSE: 10.4140 | val_MAE: 2.3473
  Epoch   8/30 | train_loss: 2.4022 | val_MSE: 10.0003 | val_MAE: 2.3068
  Epoch   9/30 | train_loss: 2.4159 | val_MSE: 9.6149 | val_MAE: 2.2663

  Early stopping at epoch 9 — best val_MSE: 8.3033
───────────────────────────────────────────────────────
Fold 2/5 | train: 403 sample

In [8]:
from torch.utils.data import DataLoader
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
result = evaluate(model, test_loader, device)
print(result)

{'loss': np.float64(4.661065888404846), 'mse': 4.82255744934082, 'mae': 1.7223666906356812, 'rmse': np.float64(2.1960322058979056), 'r2': 0.31563007831573486}
